<a href="https://www.kaggle.com/code/adegbaju/fifa-world-cup-2026-prediction-system-full?scriptVersionId=321906056" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏆 FIFA World Cup 2026 Prediction System - Full Pipeline


 **Educative grade notebook**  
> Harnessing machine learning and statistical simulation to predict match outcomes, team performance and tournament progression.

 This notebook builds a complete, reusable prediction system:
 1. **EDA** – understand the underlying team statistics
 2. **Synthetic match generation** – creates realistic training data from team snapshots
 3. **Model training & interpretation** – a classifier that predicts *win/draw/loss*
 4. **World Cup 2026 simulation** – probabilistic knockout bracket using the trained model
 5. **Interactive dashboards** – Plotly‑based visualisations for exploration


##  Setup & Library Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance

# For reproducibility
np.random.seed(42)


##  Data Loading & Inspection

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Load the main training file
train_path = '/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-prediction-system/train (1).csv'
df = pd.read_csv(train_path)

print(f"Dataset shape: {df.shape}")
df.head()

# ### Basic info
df.info()
df.describe(include='all')

# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-prediction-system/submission (17).csv
/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-prediction-system/test (2).csv
/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-prediction-system/fifa_wc2026_pipeline.py
/kaggle/input/datasets/rauffauzanrambe/fifa-world-cup-2026-prediction-system/train (1).csv
Dataset shape: (1000, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   team_name                 1000 non-null   object 
 1   country_code              1000 non-null   object 
 2   confederation             1000 non-null   object 
 3   fifa_rank                 1000 non-null   int64  
 4   fifa_points               1000 non-null   float64
 5   wins_last_10_matches      1000 non-null   int64  
 6   losses_last_10_matches    1000 non-null   in

#  Exploratory Data Analysis (EDA)


## Team & Confederation Distribution

In [3]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Confederation Distribution", "Top 15 Teams by Appearances"])
conf_counts = df['confederation'].value_counts()
fig.add_trace(go.Bar(x=conf_counts.index, y=conf_counts.values, name='Confederation'), row=1, col=1)
team_counts = df['team_name'].value_counts().head(15)
fig.add_trace(go.Bar(x=team_counts.index, y=team_counts.values, name='Team'), row=1, col=2)
fig.update_layout(height=500, showlegend=False)
fig.show()

## FIFA Rank & Points Distribution

In [4]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["FIFA Rank Distribution", "FIFA Points Distribution"])
fig.add_trace(go.Histogram(x=df['fifa_rank'], nbinsx=30, name='Rank'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['fifa_points'], nbinsx=30, name='Points'), row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig.show()

## Recent Form (wins/draws/losses last 10)

In [5]:
form_cols = ['wins_last_10_matches', 'draws_last_10_matches', 'losses_last_10_matches']
df[form_cols].sum(axis=1).describe()  # should be close to 10 for each row

fig = px.box(df.melt(value_vars=form_cols), y='value', color='variable',
             title='Distribution of Wins, Draws, Losses (Last 10 Matches)')
fig.show()

## Win Rate & Goals Scored Avg

In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Win Rate Last Year", "Goals Scored Average"])
fig.add_trace(go.Histogram(x=df['win_rate_last_year'], nbinsx=30, name='Win Rate'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['goals_scored_avg'], nbinsx=30, name='Goals Avg'), row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig.show()

# Synthetic Match Generation for Supervised Learning

In [7]:
def create_match_dataset(team_df, n_matches=10000):
    """Generate synthetic matches from team snapshots."""
    # Create a rating proxy: combine rank and points
    team_df = team_df.copy()
    # Invert rank so higher is better; scale points
    team_df['strength'] = (200 - team_df['fifa_rank']) * 0.7 + (team_df['fifa_points'] / 10) * 0.3
    
    # Store unique team identities to avoid pairing a team with itself in the same snapshot
    match_data = []
    idx = team_df.index.tolist()
    np.random.seed(42)
    
    for _ in range(n_matches):
        i, j = np.random.choice(idx, size=2, replace=False)
        row_i, row_j = team_df.iloc[i], team_df.iloc[j]
        
        # Skip if same team (though different snapshots, better avoid confusion)
        if row_i['team_name'] == row_j['team_name']:
            continue
            
        # Calculate probabilities
        rating_diff = row_j['strength'] - row_i['strength']
        p_i_win = 1 / (1 + 10**(rating_diff / 400))
        # Draw probability is roughly 25% around equal strength
        p_draw = 0.25 * np.exp(- (rating_diff / 200)**2)
        p_i_lose = 1 - p_i_win - p_draw
        
        # Sample outcome
        outcome = np.random.choice([2, 1, 0], p=[p_i_win, p_draw, p_i_lose])  # 2: win, 1: draw, 0: loss for team_i
        
        # Features: difference and absolute values
        features = {
            'rank_A': row_i['fifa_rank'],
            'rank_B': row_j['fifa_rank'],
            'points_A': row_i['fifa_points'],
            'points_B': row_j['fifa_points'],
            'win_rate_A': row_i['win_rate_last_year'],
            'win_rate_B': row_j['win_rate_last_year'],
            'goals_avg_A': row_i['goals_scored_avg'],
            'goals_avg_B': row_j['goals_scored_avg'],
            'wins_A': row_i['wins_last_10_matches'],
            'wins_B': row_j['wins_last_10_matches'],
            'losses_A': row_i['losses_last_10_matches'],
            'losses_B': row_j['losses_last_10_matches'],
            'draws_A': row_i['draws_last_10_matches'],
            'draws_B': row_j['draws_last_10_matches'],
            'conf_A': row_i['confederation'],
            'conf_B': row_j['confederation'],
            'outcome': outcome          # outcome for team A (home perspective)
        }
        match_data.append(features)
        
    return pd.DataFrame(match_data)

match_df = create_match_dataset(df, n_matches=15000)
print(f"Generated {len(match_df)} matches")
match_df.head()

# ### Class distribution
outcome_counts = match_df['outcome'].value_counts().sort_index()
outcome_labels = {0: 'Loss', 1: 'Draw', 2: 'Win'}
fig = px.bar(x=[outcome_labels[k] for k in outcome_counts.index], y=outcome_counts.values,
             labels={'x': 'Outcome', 'y': 'Count'}, title='Synthetic Match Outcome Distribution')
fig.show()


Generated 14656 matches


# Feature Engineering for the Match Model

In [8]:
# Encode confederations
conf_encoder = LabelEncoder()
match_df['conf_A_enc'] = conf_encoder.fit_transform(match_df['conf_A'])
match_df['conf_B_enc'] = conf_encoder.transform(match_df['conf_B'])

# Create difference features (from perspective of team A)
match_df['rank_diff'] = match_df['rank_B'] - match_df['rank_A']
match_df['points_diff'] = match_df['points_A'] - match_df['points_B']
match_df['win_rate_diff'] = match_df['win_rate_A'] - match_df['win_rate_B']
match_df['goals_avg_diff'] = match_df['goals_avg_A'] - match_df['goals_avg_B']
match_df['form_wins_diff'] = match_df['wins_A'] - match_df['wins_B']
match_df['form_losses_diff'] = match_df['losses_A'] - match_df['losses_B']

# Select features and target
feature_cols = ['rank_A', 'rank_B', 'points_A', 'points_B',
                'win_rate_A', 'win_rate_B', 'goals_avg_A', 'goals_avg_B',
                'wins_A', 'wins_B', 'losses_A', 'losses_B',
                'conf_A_enc', 'conf_B_enc',
                'rank_diff', 'points_diff', 'win_rate_diff', 'goals_avg_diff',
                'form_wins_diff', 'form_losses_diff']

X = match_df[feature_cols]
y = match_df['outcome']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Training set: (11724, 20), Test set: (2932, 20)


# Model Building & Comparison

In [9]:
#  training three classifiers and compare performance.

models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'model': model}
    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Loss', 'Draw', 'Win']))

# Choose best model (by accuracy)
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = results[best_model_name]['model']
print(f"\n🏆 Best model: {best_model_name} with accuracy {results[best_model_name]['accuracy']:.4f}")



RandomForest Accuracy: 0.4935
              precision    recall  f1-score   support

        Loss       0.27      0.01      0.02       757
        Draw       0.33      0.00      0.01       721
         Win       0.50      0.99      0.66      1454

    accuracy                           0.49      2932
   macro avg       0.37      0.33      0.23      2932
weighted avg       0.40      0.49      0.33      2932


GradientBoosting Accuracy: 0.4700
              precision    recall  f1-score   support

        Loss       0.27      0.06      0.10       757
        Draw       0.20      0.04      0.07       721
         Win       0.50      0.90      0.64      1454

    accuracy                           0.47      2932
   macro avg       0.32      0.33      0.27      2932
weighted avg       0.37      0.47      0.36      2932


XGBoost Accuracy: 0.4652
              precision    recall  f1-score   support

        Loss       0.30      0.09      0.13       757
        Draw       0.29      0.08    

#  Model Interpretation (SHAP alternative: Permutation Importance)


In [10]:
# SHAP might be slow; we use sklearn's permutation_importance which is model‑agnostic.

perm_importance = permutation_importance(best_model, X_test_scaled, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()[::-1]

fig = px.bar(x=[feature_cols[i] for i in sorted_idx[:15]],
             y=perm_importance.importances_mean[sorted_idx[:15]],
             labels={'x': 'Feature', 'y': 'Importance'}, title='Top 15 Feature Importances (Permutation)')
fig.show()

# World Cup 2026 Simulation


## Select the 32 most recent team snapshots (one per team)

In [11]:
team_latest = df.sort_values('fifa_rank').groupby('team_name').first().reset_index()
team_latest = team_latest.head(32)  # top 32 by FIFA rank (or some selection)
print(f"Selected {len(team_latest)} teams for the tournament")

Selected 32 teams for the tournament


## Group Draw (simulate random groups)

In [12]:
group_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
np.random.seed(2026)
shuffled_teams = team_latest.sample(frac=1).reset_index(drop=True)
groups = {grp: shuffled_teams.iloc[i*4:(i+1)*4] for i, grp in enumerate(group_names)}

# Display groups
for grp, teams in groups.items():
    print(f"Group {grp}: {', '.join(teams['team_name'].values)}")

Group A: Costa Rica, Chile, England, Ivory Coast
Group B: Iran, Japan, Colombia, Italy
Group C: Austria, Ecuador, China, Nigeria
Group D: France, Paraguay, Belgium, Egypt
Group E: Germany, Netherlands, Canada, Algeria
Group F: New Zealand, Australia, Croatia, Brazil
Group G: Ghana, Honduras, Morocco, Denmark
Group H: Jamaica, Mexico, Cameroon, Argentina


## Match Prediction Function

In [13]:
def predict_match(team_A, team_B):
    """Return probabilities [p_loss, p_draw, p_win] for team_A."""
    # Build feature vector from team snapshots
    row = pd.DataFrame({
        'rank_A': team_A['fifa_rank'],
        'rank_B': team_B['fifa_rank'],
        'points_A': team_A['fifa_points'],
        'points_B': team_B['fifa_points'],
        'win_rate_A': team_A['win_rate_last_year'],
        'win_rate_B': team_B['win_rate_last_year'],
        'goals_avg_A': team_A['goals_scored_avg'],
        'goals_avg_B': team_B['goals_scored_avg'],
        'wins_A': team_A['wins_last_10_matches'],
        'wins_B': team_B['wins_last_10_matches'],
        'losses_A': team_A['losses_last_10_matches'],
        'losses_B': team_B['losses_last_10_matches'],
        'conf_A_enc': conf_encoder.transform([team_A['confederation']])[0],
        'conf_B_enc': conf_encoder.transform([team_B['confederation']])[0],
        'rank_diff': team_B['fifa_rank'] - team_A['fifa_rank'],
        'points_diff': team_A['fifa_points'] - team_B['fifa_points'],
        'win_rate_diff': team_A['win_rate_last_year'] - team_B['win_rate_last_year'],
        'goals_avg_diff': team_A['goals_scored_avg'] - team_B['goals_scored_avg'],
        'form_wins_diff': team_A['wins_last_10_matches'] - team_B['wins_last_10_matches'],
        'form_losses_diff': team_A['losses_last_10_matches'] - team_B['losses_last_10_matches']
    }, index=[0])
    row_scaled = scaler.transform(row[feature_cols])  # ensure order matches
    probs = best_model.predict_proba(row_scaled)[0]   # [loss, draw, win] order depends on training
    return probs  # assume class order matches [0,1,2] -> loss, draw, win


## Group Stage Simulation

In [14]:
def simulate_group_stage(groups):
    standings = {}
    for grp, teams in groups.items():
        grp_teams = teams.reset_index(drop=True)
        results = []
        # Each pair plays once
        for i in range(len(grp_teams)):
            for j in range(i+1, len(grp_teams)):
                team1, team2 = grp_teams.iloc[i], grp_teams.iloc[j]
                probs = predict_match(team1, team2)
                # Sample outcome using probabilities
                outcome = np.random.choice([0,1,2], p=probs)
                if outcome == 2:   # team1 win
                    results.append({'team1': team1['team_name'], 'team2': team2['team_name'],
                                    'goals1': np.random.poisson(team1['goals_scored_avg']),
                                    'goals2': np.random.poisson(team2['goals_scored_avg'] * 0.8)})
                    # later convert to points (win=3, draw=1, loss=0)
        # This is a simplified placeholder; a full simulation would track points and goal difference.
        # For demonstration we'll just randomly assign points based on probabilities.
        # Better: repeat 1000 times Monte Carlo.
        pass
    return standings

## Interactive Bracket 

In [15]:
import random
def simulate_knockout_match(team_A, team_B):
    probs = predict_match(team_A, team_B)
    outcome = np.random.choice([0,1,2], p=probs)
    # if draw in knockout, go to extra time - simplify: re‑sample without draw
    while outcome == 1:
        probs_no_draw = np.array([probs[0], 0, probs[2]]) / (probs[0]+probs[2])
        outcome = np.random.choice([0,2], p=probs_no_draw)
    return team_A if outcome == 2 else team_B

# Create a fictional bracket with 16 teams (for visualisation)
round16_teams = list(shuffled_teams['team_name'].head(16))
bracket = []

# Interactive Dashboard with Plotly

In [16]:
fig = px.scatter(df, x='fifa_rank', y='fifa_points', color='confederation',
                 hover_data=['team_name'], title='FIFA Rank vs Points by Confederation')
fig.update_layout(height=600)
fig.show()


*Thank you for exploring this end‑to‑end FIFA World Cup 2026 prediction system!*


**DO NOT FORGET TO UPVOTE**